<a href="https://www.kaggle.com/code/poeticmage/yolo-v8-rsna-pneumonia-detection-challenge?scriptVersionId=314964646" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## RSNA Pneumonia Detection Challenge with YOLO v8
We perform this task of detecting pneumonia with YOLO from ultralytics. Before that, we obtained the data in the form of jpeg for image and txt for labels, arranged them in the required YAML format.

### Dependencies
import the necessary libraries as usual.

In [1]:
%%capture
import os
! pip install scikit-learn numpy matplotlib pandas ultralytics lightning torch 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import lightning as L
from torchvision import datasets, transforms
import pandas as pd
import numpy as np
from glob import glob
from PIL import Image
import logging
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

### YOLO
Train the model with the training data. We have split it into 7:3 training:validation

In [2]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
print("Training starts ")
model.train(
    data="/kaggle/input/datasets/poeticmage/rsna-pneumonia-detection-challenge-yaml/data.yaml",
    epochs=50,
    imgsz=640,
    verbose=False,
    project="/kaggle/working/yolo_outputs"
)
print("Training ends ")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Training starts 
Ultralytics 8.4.42 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/poeticmage/rsna-pneumonia-detection-challenge-yaml/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, frac

### Testing
Test your model, save the bounded images

In [3]:
print("Testing starts ")
results = model.predict(
    source="/kaggle/input/datasets/poeticmage/rsna-pneumonia-detection-challenge-jpeg-test/rsna_yolo/images/test",
    imgsz=768,
    conf=0.25,
    verbose=False,
    project="/kaggle/working/test_yolo_outputs",
    name="predict",
    stream=True,
    save=True
)
print("Testing ends ")

Testing starts 
Testing ends 


### Submission file
Arrange the submission file as asked.

In [4]:
from tqdm import tqdm

submission_path = os.path.join("/kaggle/working", "submission.csv")

with open(submission_path, "w") as file:
    file.write("patientId,PredictionString\n")

    for result in tqdm(results):
        patient_id = os.path.splitext(
            os.path.basename(result.path)
        )[0]

        out_str = patient_id + ","

        if result.boxes is not None and len(result.boxes) > 0:
            print("Pneumonia detected", out_str)

            for box in result.boxes:
                conf = float(box.conf[0])

                x1, y1, x2, y2 = box.xyxy[0].tolist()

                width = x2 - x1
                height = y2 - y1

                out_str += f" {round(conf, 2)} {int(x1)} {int(y1)} {int(width)} {int(height)}"

        file.write(out_str + "\n")

print("submission.csv created at:", submission_path)
print(os.listdir("/kaggle/working"))

16it [00:01, 17.26it/s]

Pneumonia detected 00330f7f-d114-4eb2-9c6e-558eeb3084a1,


22it [00:01, 17.14it/s]

Pneumonia detected 003fbda2-ba55-4714-a03a-83f15bec19e4,


43it [00:02, 23.20it/s]

...

Pneumonia detected c1937034-f8a4-4a84-a69c-213911b39907,


3000it [02:07, 24.83it/s]

Results saved to /kaggle/working/test_yolo_outputs/predict


3000it [02:07, 23.60it/s]

submission.csv created at: /kaggle/working/submission.csv
['__notebook__.ipynb', 'yolov8n.pt', 'submission.csv', 'yolo_outputs', 'test_yolo_outputs', 'yolo26n.pt']
